# Lab 01 — Cài đặt môi trường, tách từ, phân loại cảm xúc
**Môn Xử lý ngôn ngữ tự nhiên | Nhóm Ba Sáu**

**Mục lục:**
- **Bước 0** — Cài đặt môi trường
- **Bước 1** — Tải dữ liệu NLTK
- **Bước 2** — Tách từ (tokenization) và loại stopword
- **Bước 3** — Phân loại cảm xúc bằng mô hình có sẵn của Hugging Face
- **Bước 4** — Tổng kết và kiến thức cần nhớ

---
## Bước 0 — Cài đặt môi trường

- **Colab:** không cần tạo môi trường ảo, chạy thẳng lệnh `pip` bên dưới.
- **Local:** tạo môi trường ảo (`python -m venv .venv`), kích hoạt nó rồi chạy `pip install -r requirements.txt` (file nằm cạnh thư mục `notebooks/`).

In [ ]:
# ── Cell 1: Cài đặt thư viện (Colab) ──────────────────────────────────────
!pip install nltk spacy scikit-learn transformers torch

---
## Bước 1 — Tải dữ liệu NLTK

- `punkt_tab`: mô hình để NLTK **tách câu và tách từ**. Chưa tải thì `word_tokenize` báo lỗi `LookupError`.
- `stopwords`: danh sách từ dừng (the, is, and, ...) của từng ngôn ngữ, dùng ở bước loại stopword.

In [1]:
# ── Cell 2: Import và tải dữ liệu ─────────────────────────────────────────
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

---
## Bước 2 — Tách từ và loại stopword

### 2.1 Tách từ
`word_tokenize` tách một câu thành danh sách token (từ và dấu câu). Câu mẫu có thể đổi tùy ý.

In [2]:
# ── Cell 3: Tách từ bằng word_tokenize ────────────────────────────────────
from nltk.tokenize import word_tokenize

sentence = "I love studying Natural Language Processing"
tokens = word_tokenize(sentence)

print("Original sentence: ", sentence)
print("Token lists: ", tokens)
print("Số token:", len(tokens))   # kỳ vọng 6: I, love, studying, Natural, Language, Processing

Original sentence:  I love studying Natural Language Processing
Token lists:  ['I', 'love', 'studying', 'Natural', 'Language', 'Processing']
Số token: 6


### 2.2 Loại stopword
Danh sách stopword của NLTK toàn **chữ thường**, còn Python **phân biệt hoa/thường**, nên phải đưa từ về chữ thường (`.lower()`) trước khi so sánh.
Token gốc vẫn được giữ nguyên chữ hoa/thường trong kết quả.

In [3]:
# ── Cell 4: Loại stopword (có lower) ──────────────────────────────────────
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
filtered_tokens = [
    word for word in tokens
    if word.lower() not in stop_words
]
print("Sau khi loại bỏ stop words: ")
print(filtered_tokens)   # kỳ vọng: ['love', 'studying', 'Natural', 'Language', 'Processing'] ("I" bị loại)

Sau khi loại bỏ stop words: 
['love', 'studying', 'Natural', 'Language', 'Processing']


In [4]:
# ── Cell 5: Điều gì xảy ra nếu quên lower()? ──────────────────────────────
# So sánh trực tiếp, không lower(): "I" (chữ hoa) không nằm trong danh sách stopword toàn chữ thường nên không bị loại
no_lower = [word for word in tokens if word not in stop_words]
print("Không lower():", no_lower)
print("Có lower()   :", filtered_tokens)
print('"I" có trong stop_words không?', 'I' in stop_words, '| "i" có không?', 'i' in stop_words)

Không lower(): ['I', 'love', 'studying', 'Natural', 'Language', 'Processing']
Có lower()   : ['love', 'studying', 'Natural', 'Language', 'Processing']
"I" có trong stop_words không? False | "i" có không? True


---
## Bước 3 — Phân loại cảm xúc bằng mô hình có sẵn

`pipeline("sentiment-analysis")` tự tải một mô hình mặc định. Để **tái lập**, ở đây ghi rõ tên mô hình mặc định đó:
`distilbert-base-uncased-finetuned-sst-2-english` (DistilBERT fine-tune trên SST-2, chỉ có 2 nhãn POSITIVE / NEGATIVE).
Lần chạy đầu cần mạng để tải mô hình.

In [5]:
# ── Cell 6: Khởi tạo bộ phân loại ─────────────────────────────────────────
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

d:\Data Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 399.50it/s]


In [6]:
# ── Cell 7: Phân loại ba câu mẫu ──────────────────────────────────────────
# Câu mẫu đã sửa lỗi chính tả so với bản báo cáo ("so muchP", "go to shool")
texts = [
    "I love you so much",
    "I don't like going to school",
    "The book is on the table",     # câu mô tả trung tính
]
results = classifier(texts)
for text, r in zip(texts, results):
    print(f"{r['label']:<8} {r['score']:.4f}  | {text}")

POSITIVE 0.9999  | I love you so much
NEGATIVE 0.9962  | I don't like going to school
POSITIVE 0.9879  | The book is on the table


In [7]:
# ── Cell 8: Xem điểm của cả hai nhãn (câu trung tính) ─────────────────────
# top_k=None trả về điểm của TẤT CẢ nhãn; hai điểm cộng lại bằng 1
for r in classifier("The book is on the table", top_k=None):
    print(f"{r['label']:<8} {r['score']:.4f}")

POSITIVE 0.9879
NEGATIVE 0.0121


---
## Bước 4 — Tổng kết và kiến thức cần nhớ

**Kết quả quan sát trong báo cáo (Colab):**
- `word_tokenize("I love studying Natural Language Processing")` cho 6 token; sau khi loại stopword còn `['love', 'studying', 'Natural', 'Language', 'Processing']`.
- "I love you so much": POSITIVE, score ≈ 0,9999. "I don't like going to school": NEGATIVE, score ≈ 0,986.
- "The book is on the table" là câu trung tính nhưng vẫn bị gán POSITIVE với score ≈ 0,988: mô hình SST-2 chỉ có hai nhãn nên **ép** mọi câu vào positive hoặc negative.

**Kiến thức cần nhớ:**
- Phải `nltk.download('punkt_tab')` trước khi dùng `word_tokenize`.
- So sánh với stopword phải `.lower()`, vì Python phân biệt hoa/thường.
- Score của `pipeline` là độ tin cậy của **nhãn được chọn**, không phải "mức độ cảm xúc"; điểm cao không có nghĩa là mô hình đúng (xem câu trung tính ở trên).
- Không chỉ định `model=` thì Hugging Face tự chọn mô hình mặc định (và có thể đổi theo phiên bản thư viện); khi làm thật nên ghi rõ tên mô hình.
- Muốn có nhãn trung tính cần mô hình huấn luyện với 3 lớp (ví dụ trên dữ liệu ABSA như đồ án cuối kỳ), không dùng SST-2.